# Validate clustered parameter datasets

This notebook checks the five generated `c_i=(theta_i, Gamma_i)` datasets.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

ROOT = Path('.')
OUT = ROOT / 'outputs'
DATA = OUT / 'datasets'
metrics = pd.read_csv(OUT / 'metrics_summary.csv')
metrics

## Load one dataset record

In [ ]:
def load_record(path):
    with open(path, 'r') as f:
        return json.loads(f.readline())

record = load_record(DATA / 'params_K3_overlapping.jsonl')
print(record['dataset_id'])
print('labels:', record['labels'])
print('pairwise OR:', record['cluster_metrics']['pairwise_overlap_ratios'])

## SPD validation

In [ ]:
rows = []
for path in sorted(DATA.glob('params_*.jsonl')):
    r = load_record(path)
    G = np.array(r['gamma_list'])
    eig_min = min(np.linalg.eigvalsh(Gi).min() for Gi in G)
    sym_err = np.max(np.abs(G - np.swapaxes(G, 1, 2)))
    rows.append({'dataset': r['dataset_id'], 'min eigenvalue': eig_min, 'max symmetry error': sym_err})
pd.DataFrame(rows)

## Validation figures

Figures are saved under `outputs/figures/`.

In [ ]:
from IPython.display import Image, display
for name in ['params_K2_separated_mds.png', 'params_K2_overlapping_mds.png', 'params_K3_separated_mds.png', 'params_K3_overlapping_mds.png']:
    print(name)
    display(Image(filename=str(OUT / 'figures' / name)))